In [ ]:
"""
End-user LPG price allocation per pixel (Nigeria, EPSG:3857)

This notebook creates a copied output from step 4.2 and adds two columns
at pixel level:
- cost_kg_walker
- cost_kg_driver

Formula (for each pixel):
- cost_kg_walker = reseller_cost_walk + best_time_walk_min / 1000
- cost_kg_driver = reseller_cost_car + best_time_car_min / 1000

Where reseller cost is read from 4.4 output:
- dataset_big/chain_with_cost.gpkg, layer: resell, column: cost_kg

Invalid or missing assignments get sentinel value 9999.
"""

from __future__ import annotations

import sqlite3
from pathlib import Path

import fiona

DATA_DIR = Path("dataset_big")

# Input from 4.2
SOURCE_PIXEL_GPKG = DATA_DIR / "huff_preferred_distributor_per_pixel.gpkg"
PIXEL_LAYER = "pixel_preference"

# Cost source from 4.4
COST_SOURCE_GPKG = DATA_DIR / "chain_with_cost.gpkg"
COST_SOURCE_LAYER = "resell"
RESELLER_ID_COL = "id_res&fil"
RESELLER_COST_COL = "cost_kg"

# Output copy
OUTPUT_GPKG = DATA_DIR / "end_user_price.gpkg"

# Pixel columns from 4.2
WALK_ID_COL = "best_reseller_id_walk"
WALK_TIME_COL = "best_time_walk_min"
CAR_ID_COL = "best_reseller_id_car"
CAR_TIME_COL = "best_time_car_min"

# New output columns
OUT_COST_WALK = "cost_kg_walker"
OUT_COST_CAR = "cost_kg_driver"

# Sentinel and basic guards
SENTINEL = 9999.0
MAX_VALID_TIME_MIN = 7000.0

print("[1/6] Checking input files and required layers...")
if not SOURCE_PIXEL_GPKG.exists():
    raise FileNotFoundError(f"Missing input GPKG from 4.2: {SOURCE_PIXEL_GPKG}")
if not COST_SOURCE_GPKG.exists():
    raise FileNotFoundError(f"Missing cost source GPKG from 4.4: {COST_SOURCE_GPKG}")

pixel_layers = fiona.listlayers(str(SOURCE_PIXEL_GPKG))
if PIXEL_LAYER not in pixel_layers:
    raise KeyError(f"Layer '{PIXEL_LAYER}' not found in {SOURCE_PIXEL_GPKG}")

cost_layers = fiona.listlayers(str(COST_SOURCE_GPKG))
if COST_SOURCE_LAYER not in cost_layers:
    raise KeyError(f"Layer '{COST_SOURCE_LAYER}' not found in {COST_SOURCE_GPKG}")

print("[2/6] Creating working copy end_user_price.gpkg...")
if OUTPUT_GPKG.exists():
    OUTPUT_GPKG.unlink()
OUTPUT_GPKG.write_bytes(SOURCE_PIXEL_GPKG.read_bytes())

print("[3/6] Validating schema in copied output and source cost layer...")
with sqlite3.connect(str(OUTPUT_GPKG)) as conn_out:
    cur_out = conn_out.cursor()
    pixel_cols = [c[1] for c in cur_out.execute(f'PRAGMA table_info("{PIXEL_LAYER}")').fetchall()]

for c in [WALK_ID_COL, WALK_TIME_COL, CAR_ID_COL, CAR_TIME_COL]:
    if c not in pixel_cols:
        raise KeyError(f"Missing required pixel column '{c}' in layer '{PIXEL_LAYER}'")

with sqlite3.connect(str(COST_SOURCE_GPKG)) as conn_cost:
    cur_cost = conn_cost.cursor()
    cost_cols = [c[1] for c in cur_cost.execute(f'PRAGMA table_info("{COST_SOURCE_LAYER}")').fetchall()]

for c in [RESELLER_ID_COL, RESELLER_COST_COL]:
    if c not in cost_cols:
        raise KeyError(f"Missing required cost column '{c}' in layer '{COST_SOURCE_LAYER}'")

print("[4/6] Adding output columns to copied GPKG (if needed)...")
with sqlite3.connect(str(OUTPUT_GPKG)) as conn_out:
    cur = conn_out.cursor()
    cols = [c[1] for c in cur.execute(f'PRAGMA table_info("{PIXEL_LAYER}")').fetchall()]
    if OUT_COST_WALK not in cols:
        cur.execute(f'ALTER TABLE "{PIXEL_LAYER}" ADD COLUMN "{OUT_COST_WALK}" REAL')
    if OUT_COST_CAR not in cols:
        cur.execute(f'ALTER TABLE "{PIXEL_LAYER}" ADD COLUMN "{OUT_COST_CAR}" REAL')
    conn_out.commit()

print("[5/6] Computing end-user prices for walk and car...")
attach_sql = "ATTACH DATABASE ? AS src"

with sqlite3.connect(str(OUTPUT_GPKG)) as conn_out:
    cur = conn_out.cursor()
    cur.execute(attach_sql, (str(COST_SOURCE_GPKG),))

    update_sql = f'''
    UPDATE "{PIXEL_LAYER}"
    SET
        "{OUT_COST_WALK}" = CASE
            WHEN "{WALK_ID_COL}" > 0
             AND "{WALK_TIME_COL}" IS NOT NULL
             AND "{WALK_TIME_COL}" >= 0
             AND "{WALK_TIME_COL}" < {MAX_VALID_TIME_MIN}
             AND (
                SELECT CAST(r."{RESELLER_COST_COL}" AS REAL)
                FROM src."{COST_SOURCE_LAYER}" AS r
                WHERE CAST(r."{RESELLER_ID_COL}" AS INTEGER) = CAST("{PIXEL_LAYER}"."{WALK_ID_COL}" AS INTEGER)
                LIMIT 1
             ) IS NOT NULL
            THEN (
                SELECT CAST(r."{RESELLER_COST_COL}" AS REAL)
                FROM src."{COST_SOURCE_LAYER}" AS r
                WHERE CAST(r."{RESELLER_ID_COL}" AS INTEGER) = CAST("{PIXEL_LAYER}"."{WALK_ID_COL}" AS INTEGER)
                LIMIT 1
            ) + CAST("{WALK_TIME_COL}" AS REAL) / 1000.0
            ELSE {SENTINEL}
        END,

        "{OUT_COST_CAR}" = CASE
            WHEN "{CAR_ID_COL}" > 0
             AND "{CAR_TIME_COL}" IS NOT NULL
             AND "{CAR_TIME_COL}" >= 0
             AND "{CAR_TIME_COL}" < {MAX_VALID_TIME_MIN}
             AND (
                SELECT CAST(r."{RESELLER_COST_COL}" AS REAL)
                FROM src."{COST_SOURCE_LAYER}" AS r
                WHERE CAST(r."{RESELLER_ID_COL}" AS INTEGER) = CAST("{PIXEL_LAYER}"."{CAR_ID_COL}" AS INTEGER)
                LIMIT 1
             ) IS NOT NULL
            THEN (
                SELECT CAST(r."{RESELLER_COST_COL}" AS REAL)
                FROM src."{COST_SOURCE_LAYER}" AS r
                WHERE CAST(r."{RESELLER_ID_COL}" AS INTEGER) = CAST("{PIXEL_LAYER}"."{CAR_ID_COL}" AS INTEGER)
                LIMIT 1
            ) + CAST("{CAR_TIME_COL}" AS REAL) / 1000.0
            ELSE {SENTINEL}
        END
    '''

    cur.execute(update_sql)
    updated_rows = cur.rowcount
    cur.execute("DETACH DATABASE src")
    conn_out.commit()

print("[6/6] Done.")
print(f"Output file: {OUTPUT_GPKG}")
print(f"Updated rows in layer '{PIXEL_LAYER}': {updated_rows:,}")
print(f"Columns added/updated: {OUT_COST_WALK}, {OUT_COST_CAR}")
print("Formula used:")
print("  cost_kg_walker = reseller_cost_walk + best_time_walk_min / 1000")
print("  cost_kg_driver = reseller_cost_car + best_time_car_min / 1000")
print(f"Sentinel for invalid assignments: {SENTINEL}")

In [ ]:
"""
Final QA stats for 4.5 output.

Checks:
1) output file and layer exist
2) new columns were populated
3) valid vs sentinel counts for walker/driver
4) distribution stats on valid costs
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import fiona

DATA_DIR = Path("dataset_big")
OUTPUT_GPKG = DATA_DIR / "end_user_price.gpkg"
PIXEL_LAYER = "pixel_preference"

WALK_ID_COL = "best_reseller_id_walk"
WALK_TIME_COL = "best_time_walk_min"
CAR_ID_COL = "best_reseller_id_car"
CAR_TIME_COL = "best_time_car_min"
OUT_COST_WALK = "cost_kg_walker"
OUT_COST_CAR = "cost_kg_driver"
SENTINEL = 9999.0

print("[1/4] Loading output layer...")
if not OUTPUT_GPKG.exists():
    raise FileNotFoundError(f"Output file not found: {OUTPUT_GPKG}")

layers = fiona.listlayers(str(OUTPUT_GPKG))
if PIXEL_LAYER not in layers:
    raise KeyError(f"Layer '{PIXEL_LAYER}' not found in {OUTPUT_GPKG}")

gdf = gpd.read_file(OUTPUT_GPKG, layer=PIXEL_LAYER)
if gdf.empty:
    raise RuntimeError("Output layer is empty.")

required = [WALK_ID_COL, WALK_TIME_COL, CAR_ID_COL, CAR_TIME_COL, OUT_COST_WALK, OUT_COST_CAR]
missing = [c for c in required if c not in gdf.columns]
if missing:
    raise KeyError(f"Missing required columns in output layer: {missing}")

def _num_array(col_name: str) -> np.ndarray:
    values = pd.to_numeric(pd.Series(gdf[col_name]), errors="coerce")
    return np.asarray(values, dtype=np.float64)

print("[2/4] Preparing numeric arrays...")
walk_id = _num_array(WALK_ID_COL)
walk_time = _num_array(WALK_TIME_COL)
car_id = _num_array(CAR_ID_COL)
car_time = _num_array(CAR_TIME_COL)
cost_walk = _num_array(OUT_COST_WALK)
cost_car = _num_array(OUT_COST_CAR)

print("[3/4] QA checks and summary metrics...")
n = len(gdf)

walk_finite = np.isfinite(cost_walk)
car_finite = np.isfinite(cost_car)

walk_sentinel = walk_finite & np.isclose(cost_walk, SENTINEL)
car_sentinel = car_finite & np.isclose(cost_car, SENTINEL)

walk_valid = walk_finite & (~walk_sentinel)
car_valid = car_finite & (~car_sentinel)

walk_formula_positive = walk_valid & (cost_walk >= (walk_time / 1000.0))
car_formula_positive = car_valid & (cost_car >= (car_time / 1000.0))

print("=== END USER PRICE QA ===")
print(f"Rows total                              : {n:,}")
print(f"Walker finite cost                      : {int(walk_finite.sum()):,}/{n:,}")
print(f"Driver finite cost                      : {int(car_finite.sum()):,}/{n:,}")
print(f"Walker valid (cost != {SENTINEL:.0f})            : {int(walk_valid.sum()):,}/{n:,}")
print(f"Driver valid (cost != {SENTINEL:.0f})            : {int(car_valid.sum()):,}/{n:,}")
print(f"Walker sentinel ({SENTINEL:.0f}) count           : {int(walk_sentinel.sum()):,}/{n:,}")
print(f"Driver sentinel ({SENTINEL:.0f}) count           : {int(car_sentinel.sum()):,}/{n:,}")

print("\n=== INPUT LINKAGE SANITY ===")
print(f"Rows with walk reseller id > 0           : {int((walk_id > 0).sum()):,}/{n:,}")
print(f"Rows with car reseller id > 0            : {int((car_id > 0).sum()):,}/{n:,}")
print(f"Rows with finite walk time               : {int(np.isfinite(walk_time).sum()):,}/{n:,}")
print(f"Rows with finite car time                : {int(np.isfinite(car_time).sum()):,}/{n:,}")

if walk_valid.any():
    w = cost_walk[walk_valid]
    print("\nWalker cost min/median/mean/p95/max:",
          f"{np.nanmin(w):.4f} / {np.nanmedian(w):.4f} / {np.nanmean(w):.4f} / {np.nanpercentile(w, 95):.4f} / {np.nanmax(w):.4f}")

if car_valid.any():
    c = cost_car[car_valid]
    print("Driver cost min/median/mean/p95/max:",
          f"{np.nanmin(c):.4f} / {np.nanmedian(c):.4f} / {np.nanmean(c):.4f} / {np.nanpercentile(c, 95):.4f} / {np.nanmax(c):.4f}")

print("\nFormula consistency quick check (valid rows):")
print(f"Walker rows with cost >= time/1000      : {int(walk_formula_positive.sum()):,}/{int(walk_valid.sum()):,}")
print(f"Driver rows with cost >= time/1000      : {int(car_formula_positive.sum()):,}/{int(car_valid.sum()):,}")

print("[4/4] QA completed.")
print(f"Output checked: {OUTPUT_GPKG} | layer={PIXEL_LAYER}")